# Scaling Analysis

Analyze how model properties scale with depth and width: capacity utilization, feature density, component saturation, contribution profiles, and representation compression.

## Why This Matters

Scaling scaling investigates how model properties change with size. Understanding scaling trends — does a circuit become more or less important in larger models? — is essential for determining whether interpretability findings generalize beyond the specific model studied.

**Key references:**
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.scaling_analysis import (
    layer_capacity_utilization, feature_density,
    component_saturation, depth_contribution_profile,
    representation_compression,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
result = layer_capacity_utilization(model, tokens)
print(f'Mean effective rank: {result["mean_effective_rank"]}')
for p in result['per_layer']:
    print(f'  Layer {p["layer"]}: eff_rank={p["effective_rank"]}, util={p["utilization"]:.2%}')

In [ ]:
tokens_list = [jnp.array([i, i+10, i+20, i+30, i+40]) for i in range(1, 30)]
result = feature_density(model, tokens_list, layer=-1)
print(f'Feature density estimate: {result["density_estimate"]}')
print(f'Bimodal directions: {result["n_bimodal"]}/{result["n_directions"]}')

In [ ]:
result = component_saturation(model, tokens)
for p in result['per_layer']:
    print(f'Layer {p["layer"]}: attn_max={p["attn_mean_max"]:.3f}, '
          f'dead_frac={p["mlp_dead_fraction"]:.3f}')

In [ ]:
result = depth_contribution_profile(model, tokens)
for p in result['per_layer']:
    print(f'Layer {p["layer"]}: resid={p["residual_norm"]:.3f}, '
          f'attn={p["attn_contribution_norm"]:.3f}, mlp={p["mlp_contribution_norm"]:.3f}')

In [ ]:
result = representation_compression(model, tokens_list[:10])
for p in result['per_layer']:
    print(f'Layer {p["layer"]}: eff_dim={p["effective_dimensionality"]:.1f}, '
          f'mean_dist={p["mean_pairwise_distance"]:.3f}')